# Escape Project: Engineering Notebook

**Image-based pool detection, fuel-constrained route planning, and numerical analysis in C**

This notebook is the central technical reference for the project. It reconstructs the system from the verified C implementation and the checked-in sample data. Each computational topic follows the sequence **theory → implementation → visual result → interpretation**.

> Evidence labels used below: **Verified** means checked directly against the current code or data; **Recomputed** means derived from the canonical sample; **Supplementary explanation** means engineering context added to make the implementation easier to understand.

## 1. Problem and system overview

The system models an amphibious field robot moving on a horizontal two-dimensional map. Fish pools are represented by a specific color in an uncompressed 24-bit BMP image. A pool can replenish the robot's fuel, but extracting fuel consumes time. The planning task is therefore a resource-constrained minimum-time search rather than a geometric shortest path alone.

The C program provides five operations:

1. detect pools and export their geometry;
2. sort the detected pools by area;
3. find and render a feasible escape route;
4. integrate a nonlinear cumulative-cost model; and
5. plan and price a greedy fishing trip.

![Project processing pipeline](../assets/diagrams/project-overview.svg)

## 2. Requirements, runtime boundary, and architecture

### Functional requirements

| Operation | Required input | Primary output | Engineering purpose |
|---|---|---|---|
| Pool scan | 24-bit BMP map | pool centers and areas | convert pixels into geometric data |
| Pool sort | detected-pool table | descending area list | inspect and rank available resources |
| Escape route | start point, initial fuel, pool table | best-route table and BMP overlay | minimize elapsed time subject to fuel feasibility |
| Numerical report | best-route table, display interval | cost-profile CSV | evaluate the nonlinear cumulative-cost model |
| Fishing route | requested quantity, pool table | closed BMP tour and price summary | demonstrate a second planning policy on the same map |

### Runtime boundary

**Verified.** This repository is an offline C11 simulation and analysis program. It has no physical sensors, actuators, embedded controller, network service, or graphical user interface. The modeled robot and its fuel behavior are engineering abstractions; all observable inputs are local files or console values, and all outputs are text, CSV, console, or BMP artifacts.

The implementation depends only on the C standard library and `libm`. A C11 compiler such as GCC or Clang is required. The convenience build uses GNU Make; no third-party runtime library is required.

### Architecture and core data structures

| Structure | Responsibility | Important fields |
|---|---|---|
| `Image` | normalized in-memory BMP | width, height, RGB pixel array |
| `Pool` / `PoolData` | detected resource geometry | center, area, map dimensions |
| `RouteStop` | one planning state on a route | point, pool area, time, fuel |
| `Route` | ordered candidate or best route | dynamic stop array, total time |
| `SearchContext` | recursive-search working state | pool table, destination, visited set, current and best routes |

```c
typedef struct {
    Point point;
    int pool_size;
    double time;
    double fuel;
} RouteStop;
```

Data flows in one direction: the canonical BMP produces the pool table; that table drives both route planners; the escape route drives the numerical model. This separation prevents a documented result from being paired with unrelated input data.

## 3. Canonical input and coordinate system

**Verified.** The canonical image is `examples/input/fishpool.bmp`, an uncompressed 24-bit BMP with dimensions $13\times15$. The lower-left pixel is coordinate $(1,1)$; the upper-right destination is $(13,15)$. One pixel represents one square meter.

The BMP stores each pixel as BGR bytes and pads every row to a multiple of four bytes. For width $W$, the stored row length is

$$B_{row}=4\left\lceil\frac{3W}{4}\right\rceil\text{ bytes}.$$

The implementation normalizes those bytes into an in-memory RGB array indexed in bottom-left Cartesian coordinates. A positive BMP height means the file already stores rows bottom-up; a negative height is converted from top-down order.

![Nearest-neighbor enlargement of the actual input pixels](../assets/results/input-map.png)

In [1]:
# Independent inspection using only Python's standard library.
from pathlib import Path
import struct

bmp_path = Path('../examples/input/fishpool.bmp')
header = bmp_path.read_bytes()[:54]
width, height = struct.unpack_from('<ii', header, 18)
bits_per_pixel = struct.unpack_from('<H', header, 28)[0]
compression = struct.unpack_from('<I', header, 30)[0]
print('BMP signature:', header[:2].decode('ascii'))
print(f'Dimensions: {width} x {height}')
print('Bits per pixel:', bits_per_pixel)
print('Compression:', compression)

BMP signature: BM
Dimensions: 13 x 15
Bits per pixel: 24
Compression: 0


## 4. Pool detection

### Theory

A pixel belongs to a candidate pool only when its RGB value is exactly $(155,190,245)$. Candidate pixels are grouped with four-neighbor connectivity. A connected component is accepted as a pool when it contains at least ten pixels.

For an accepted component, its area is the number of pixels $A$, and its representative center is the integer midpoint of its bounding box. Here $x_{min}$, $x_{max}$, $y_{min}$, and $y_{max}$ are the extreme one-based pixel coordinates of that component:

$$x_c=\left\lfloor\frac{x_{min}+x_{max}}{2}\right\rfloor,\qquad y_c=\left\lfloor\frac{y_{min}+y_{max}}{2}\right\rfloor.$$

### C implementation

The implementation performs an iterative breadth-first traversal. The explicit queue avoids recursion depth limits and every pixel is marked once, giving $O(WH)$ time and $O(WH)$ auxiliary memory for a $W\times H$ image.

```c
if (!visited[neighbor] && is_pool_pixel(image_pixel_const(image, nx, ny))) {
    visited[neighbor] = 1;
    queue[tail++] = (int)neighbor;
}

if (count >= MIN_POOL_PIXELS) {
    Pool pool = {{(min_x + max_x) / 2,
                  (min_y + max_y) / 2}, count};
    append_pool(data, pool);
}
```

### Result and interpretation

**Recomputed.** The sample contains three blue components with 21, 23, and 5 pixels. The five-pixel component is correctly rejected. The two retained pools are $(9,3)$ with area $21\,m^2$ and $(4,12)$ with area $23\,m^2$.

In [2]:
# Read the deterministic output produced by `escape --scan`.
import re

lines = Path('../examples/expected/detected-pools.txt').read_text().splitlines()
map_size = tuple(map(int, re.findall(r'\d+', lines[0])))
records = [tuple(map(int, re.findall(r'\d+', line))) for line in lines[3:]]
print(f'Map: {map_size[0]} x {map_size[1]}')
for index, (x, y, area) in enumerate(records, 1):
    print(f'Pool {index}: center=({x},{y}), area={area} m^2')

Map: 13 x 15
Pool 1: center=(9,3), area=21 m^2
Pool 2: center=(4,12), area=23 m^2


## 5. Pool-data contract and sorting

### File contract

The scanner writes a human-readable table whose first line fixes the map dimensions and whose remaining records contain `(x,y)` and area. The route and fishing commands validate every coordinate against those dimensions and reject pool areas below the detection threshold. This makes a mismatched image/table pair fail explicitly instead of allowing an out-of-range image access.

```text
Image size (13x15)
Pool Center    Size
(9,3)          21
(4,12)         23
```

### Sorting implementation and result

The pool list is sorted by descending area with the standard-library `qsort` function. Coordinate tie-breakers make the output deterministic. For $P$ pools this operation takes $O(P\log P)$ average time and $O(P)$ storage for the pool table.

```c
qsort(data->items, data->count, sizeof(*data->items),
      compare_pool_size_desc);
```

**Verified.** The canonical descending order is $(4,12)$ with area 23, followed by $(9,3)$ with area 21.

## 6. Fuel and time model

### Theory

The Euclidean distance between two states is

$$d(P,Q)=\sqrt{(x_Q-x_P)^2+(y_Q-y_P)^2}.$$

The verified model parameters are:

| Symbol | Value or unit | Meaning |
|---|---:|---|
| $v$ | $0.2\,m/s$ | robot speed |
| $\alpha_o$ | $0.2\,cm^3/m$ | movement fuel consumption |
| $\delta$ | $0.2\,cm^3/s$ | pool fuel extraction rate |
| $d$ | meters | Euclidean segment length; one pixel equals one meter |
| $A$ | $m^2$ and numerically seconds | pool area and mandated extraction duration |
| $t$ | seconds | elapsed route time |
| $O$ | $cm^3$ | remaining fuel volume |

A move of distance $d$ is feasible when $O\ge0.2d$. Moving to a pool of area $A$ updates the state as

$$t'=t+5d+A,\qquad O'=O-0.2d+0.2A.$$

Moving directly to the destination omits the extraction terms. For route points $r_0,\ldots,r_n$ with intermediate pool areas $A_1,\ldots,A_{n-1}$, the total modeled time is

$$T(R)=5\sum_{i=0}^{n-1}d(r_i,r_{i+1})+\sum_{i=1}^{n-1}A_i.$$

Fuel evolves with the recurrence above, and every segment must be feasible before it is taken. The route objective is therefore

$$R^*=\operatorname*{arg\,min}_{R\in\mathcal{F}}T(R),$$

where $\mathcal{F}$ is the set of routes satisfying the fuel and neighbor constraints.

### C implementation

```c
if (current.fuel + 1e-9 >= MOVE_FUEL_PER_M * distance) {
    stop.time = current.time + distance / MOVE_SPEED_MPS + pool->size;
    stop.fuel = current.fuel - MOVE_FUEL_PER_M * distance
              + EXTRACTION_FUEL_PER_S * pool->size;
}
```

## 7. Recursive route search

### Algorithm

At each state, the search first tests a direct move to the destination. It then recursively explores the nearest and second-nearest unvisited pools when enough fuel is available. The state carries the current route and a visited-pool set; backtracking restores both after each branch. Every feasible destination route is compared by elapsed time.

This is a constrained depth-first enumeration, not Dijkstra's algorithm: fuel changes at pools, and the assignment limits each expansion to two nearby choices. In the worst case the number of branches is exponential in the number of pools.

![Route-search state and transitions](../assets/diagrams/route-search.svg)

### Verified result

Starting at $(1,1)$ with $3.0\,cm^3$, the destination is initially out of range. The best feasible route is

$$ (1,1)\rightarrow(9,3)\rightarrow(13,15). $$

It takes $125.476609\,s$ and finishes with $3.020936\,cm^3$. The lower pool supplies enough fuel for the final segment.

![Verified route overlay](../assets/results/route-overlay.png)

In [3]:
route_lines = Path('../examples/expected/best-route.txt').read_text().splitlines()[3:]
for line in route_lines:
    x, y, size, time, fuel = re.findall(r'-?\d+(?:\.\d+)?', line)
    print(f'({x},{y}) size={size} time={time} fuel={fuel}')

(1,1) size=0 time=0.000000 fuel=3.000000
(9,3) size=21 time=62.231056 fuel=5.550758
(13,15) size=0 time=125.476609 fuel=3.020936


## 8. Rasterizing the route

### Theory and implementation

The route is a polyline through integer grid coordinates. Each segment is rasterized with Bresenham's line algorithm, which uses integer error accumulation and writes only validated image coordinates. This replaces slope-based formulas that require separate vertical-line cases and can divide by zero.

```c
for (;;) {
    Pixel *pixel = image_pixel(image, x, y);
    if (pixel) *pixel = color;
    if (x == end.x && y == end.y) break;
    if (2 * error >= dy) { error += dy; x += sx; }
    if (2 * error <= dx) { error += dx; y += sy; }
}
```

The overlay uses red for the start, yellow for a pool stop, green for the destination, and blue for travel. Because the source is only $13\times15$ pixels, the documentation shows a nearest-neighbor enlargement; the checked-in BMP remains the exact computational output.

## 9. Numerical cost model

### Theory

The cumulative cost satisfies

$$\frac{dc}{dx}=\frac{2.5}{c+1}+F(x),\qquad c(0)=0,$$

where $c$ is cumulative model cost, $x$ is route progress in meters (equivalently pixels in this map), and $F(x)$ is the action term: $F=1$ during movement and $F=20$ at a fuel-extraction event. The coefficient 2.5 is the fixed nonlinear cost coefficient. Cost values are model units rather than measured currency or energy. Forward Euler with $\Delta x=0.1\,m$ gives

$$c_{k+1}=c_k+\Delta x\left(\frac{2.5}{c_k+1}+F_k\right).$$

### C implementation

```c
static double euler_step(double cost, double action_cost, double dx) {
    return cost + dx * (COST_ALPHA / (cost + 1.0) + action_cost);
}
```

### Result and interpretation

**Recomputed.** Along the verified $20.90\,m$ route, the integrated cost reaches $28.176$. The orange point is the discrete high-cost extraction event at the intermediate pool. The derivative decreases gradually as $c$ grows because $2.5/(c+1)$ becomes smaller; movement still contributes the constant $F=1$.

![Numerical cost profile](../assets/results/cost-profile.png)

In [4]:
import csv

with Path('../examples/expected/cost-profile.csv').open(newline='') as stream:
    samples = list(csv.DictReader(stream))
print('Samples:', len(samples))
print(f"Final distance: {float(samples[-1]['distance_m']):.6f} m")
print(f"Final cost: {float(samples[-1]['cost']):.6f}")
print('Extraction events:', sum(row['phase'] == 'extraction' for row in samples))

Samples: 213
Final distance: 20.895322 m
Final cost: 28.176332
Extraction events: 1


## 10. Fishing extension

### Theory and algorithm

For $P$ detected pools, the logistics point is the component-wise rounded mean of their centers:

$$L=\left(\operatorname{round}\left(\frac{1}{P}\sum_{i=1}^{P}x_i\right),\operatorname{round}\left(\frac{1}{P}\sum_{i=1}^{P}y_i\right)\right).$$

Starting at $L$, the algorithm repeatedly visits the nearest unvisited pool until the accumulated capacity satisfies $\sum_{i\in V}A_i\ge Q$, then returns to $L$. The closed-tour distance is

$$D=d(L,p_1)+\sum_{i=1}^{k-1}d(p_i,p_{i+1})+d(p_k,L).$$

This is the nearest-neighbor heuristic: simple and deterministic, but not globally optimal. A direct implementation scans the remaining pools at each stop, giving $O(P^2)$ time and $O(P)$ route storage.

For requested quantity $Q$ in fish and round-trip distance $D$ in meters, prices are expressed in the model's currency units:

$$P_{fish}=\begin{cases}Q,&Q<100\\0.75Q,&Q\ge100,\end{cases}\qquad P_{fuel}=0.2D,\qquad P_{total}=P_{fish}+P_{fuel}.$$

### C implementation

```c
while (collected < requested) {
    int best = -1;
    double best_distance = HUGE_VAL;
    for (size_t i = 0; i < data.count; ++i) {
        double candidate;
        if (visited[i]) continue;
        candidate = point_distance(current, data.items[i].center);
        if (candidate < best_distance) {
            best_distance = candidate;
            best = (int)i;
        }
    }
    if (best < 0) goto cleanup;
    distance += best_distance;
    current = data.items[best].center;
    collected += data.items[best].size;
    visited[best] = 1;
}
distance += point_distance(current, logistic);
```

### Verified sample

For a 30-fish order, the logistics point is $(7,8)$. Both pools are visited, providing capacity 44. The closed route length is $20.681\,m$, the fuel price is $4.136$, and the total price is $34.136$. The canonical sample cannot exercise the discount branch because its total capacity is only 44.

![Greedy fishing route](../assets/results/fishing-route.png)

### Reported legacy demonstrations

The following real console results were documented for three larger maps. Their source BMP and pool-table files are not available in the canonical dataset, so the values are preserved as **reported** rather than presented as reproducible or recomputed results.

| Map size | Available capacity | Order | Fish price | Fuel price | Total price |
|---:|---:|---:|---:|---:|---:|
| 275 x 305 | 3429 | 1000 | 750.000 | 50.814 | 800.814 |
| 105 x 80 | 283 | 200 | 150.000 | 16.272 | 166.272 |
| 150 x 140 | 437 | 90 | 90.000 | 22.566 | 112.566 |

The 1000- and 200-fish rows exercise the 25% discount branch; the 90-fish row does not. No missing route length or map geometry is inferred from these price totals.

## 11. Verification matrix and engineering workflow

The checked-in values are reproducible demonstrations of the canonical sample, not measurements from physical hardware. No synthetic result is presented as an experiment.

| Stage | Source input | Checked artifact | Verified observation |
|---|---|---|---|
| BMP inspection | `fishpool.bmp` | independent header read | 13 x 15, 24-bit, uncompressed |
| Pool detection | canonical BMP | `detected-pools.txt` | two valid pools; one five-pixel component rejected |
| Sorting | detected-pool table | console output | areas 23 then 21 |
| Escape planning | start `(1,1)`, fuel 3 | `best-route.txt`, route BMP | 125.476609 s; 3.020936 cm^3 remaining |
| Numerical model | best-route table | `cost-profile.csv` | 20.895322 m; final cost 28.176332 |
| Fishing extension | order 30 | fishing-route BMP, console summary | 20.681 m closed tour; total price 34.136 |

### Verification workflow

1. Compile the C source with strict warnings enabled.
2. Generate the pool table from the canonical BMP rather than pairing independent sample files.
3. Feed that table into sorting and both planners.
4. Feed the serialized escape route into the numerical model.
5. Validate notebook JSON, local links, BMP headers, failure paths, and the absence of personal or institutional metadata.

### Complexity summary

| Operation | Time | Auxiliary storage |
|---|---:|---:|
| BMP scan and connected components | $O(WH)$ | $O(WH)$ |
| Pool sorting | $O(P\log P)$ average | $O(P)$ data table |
| Two-branch recursive route search | $O(2^P)$ worst case | $O(P)$ recursion depth plus saved routes |
| Euler integration | $O(D/\Delta x)$ | $O(1)$ working state plus CSV output |
| Greedy fishing route | $O(P^2)$ | $O(P)$ |

Here $W$ and $H$ are image dimensions, $P$ is the number of valid pools, and $D$ is route length in meters.

## 12. Reproduction guide

From the repository root:

```bash
make
make demo
```

The demo target executes the same sequence used to create the checked-in results:

```bash
./build/escape --scan
./build/escape --sort
./build/escape --route 1,1 3
./build/escape --cost 5
./build/escape --fish 30
```

Each command also accepts explicit input and output paths; run `./build/escape --help` for the full interface.

## 13. Limitations and engineering conclusions

- Exact-color segmentation is appropriate for generated maps but sensitive to antialiasing, compression, and color drift.
- The BMP parser deliberately supports only uncompressed 24-bit images. This keeps the binary format handling explicit and testable.
- Expanding two nearest unvisited pools implements the stated graph constraint but can exclude a feasible route that would exist in a fully connected graph.
- Recursive enumeration can grow exponentially. A larger production system should use dominance pruning on `(position, visited, fuel, time)` states or a resource-constrained shortest-path method.
- The numerical extraction model is represented as a discrete Euler event at a pool. This convention is explicit in the CSV and should be revisited if extraction is to be parameterized by duration or distance.
- The fishing extension optimizes neither distance nor price globally. Its value is as a transparent application of the same geometry and image-rendering pipeline.

The central engineering result is a single reproducible data flow: the canonical BMP generates the pool table, which drives both planning algorithms and all displayed results. No documented measurement or route is detached from its source input.